import mysql connector

In [2]:
import mysql.connector
import pandas as pd

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root",
    database="ecommerce_db"
)

cursor = conn.cursor()

print("MySQL connected successfully")

MySQL connected successfully


create total_order_values

In [3]:
query = """
SELECT
    order_id,
    SUM(payment_value) AS total_order_value
FROM order_payments
GROUP BY order_id
"""

order_value_df = pd.read_sql(query, conn)

order_value_df.head()

C:\Users\Abcom\AppData\Local\Temp\ipykernel_724\178819445.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  order_value_df = pd.read_sql(query, conn)


,order_id,total_order_value
0,00010242fe8c5a6d1ba2dd792cb16214,72.19
1,00018f77f2f0320c557190d7a144bdd3,259.83
2,000229ec398224ef6ca0657da4fc703e,216.87
3,00024acbcdf0a6daa1e931b038114c75,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04


check the feature

In [4]:
print("Rows:", len(order_value_df))
print("Columns:", order_value_df.columns.tolist())
print(order_value_df["total_order_value"].describe())

Rows: 99438
Columns: ['order_id', 'total_order_value']
count    99438.000000
mean       160.991608
std        221.953230
min          0.000000
25%         62.010000
50%        105.290000
75%        176.970000
max      13664.080000
Name: total_order_value, dtype: float64


Feature 2 — Delivery Days

Delivery Days = order_delivered_customer_date − order_purchase_timestamp

In [5]:
query = """
SELECT
    order_id,
    order_purchase_timestamp,
    order_delivered_customer_date
FROM orders
WHERE order_purchase_timestamp IS NOT NULL
  AND order_delivered_customer_date IS NOT NULL
"""

delivery_df = pd.read_sql(query, conn)

delivery_df["delivery_days"] = (
    delivery_df["order_delivered_customer_date"]
    - delivery_df["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

delivery_df.head()

C:\Users\Abcom\AppData\Local\Temp\ipykernel_724\1708072691.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  delivery_df = pd.read_sql(query, conn)


,order_id,order_purchase_timestamp,order_delivered_customer_date,delivery_days
0,00010242fe8c5a6d1ba2dd792cb16214,2017-09-13 08:59:02,2017-09-20 23:43:48,7.614421
1,00018f77f2f0320c557190d7a144bdd3,2017-04-26 10:53:06,2017-05-12 16:04:24,16.216181
2,000229ec398224ef6ca0657da4fc703e,2018-01-14 14:33:31,2018-01-22 13:19:16,7.948437
3,00024acbcdf0a6daa1e931b038114c75,2018-08-08 10:00:35,2018-08-14 13:32:39,6.147269
4,00042b26cf59d7ce69dfabb4e55b4fd9,2017-02-04 13:57:51,2017-03-01 16:42:31,25.114352


In [6]:
print("Rows:", len(delivery_df))
print("Columns:", delivery_df.columns.tolist())
print(delivery_df["delivery_days"].describe())

Rows: 96476
Columns: ['order_id', 'order_purchase_timestamp', 'order_delivered_customer_date', 'delivery_days']
count    96476.000000
mean        12.558702
std          9.546530
min          0.533414
25%          6.766403
50%         10.217755
75%         15.720327
max        209.628611
Name: delivery_days, dtype: float64


Feature 3 — Delivery Delay

Delivery Delay = Actual Delivery Date − Estimated Delivery Date

In [7]:
query = """
SELECT
    order_id,
    order_delivered_customer_date,
    order_estimated_delivery_date
FROM orders
WHERE order_delivered_customer_date IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL
"""

delay_df = pd.read_sql(query, conn)

delay_df["delivery_delay"] = (
    delay_df["order_delivered_customer_date"]
    - delay_df["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 60 * 60)

delay_df.head()

C:\Users\Abcom\AppData\Local\Temp\ipykernel_724\1186817276.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  delay_df = pd.read_sql(query, conn)


,order_id,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay
0,00010242fe8c5a6d1ba2dd792cb16214,2017-09-20 23:43:48,2017-09-29,-8.011250
1,00018f77f2f0320c557190d7a144bdd3,2017-05-12 16:04:24,2017-05-15,-2.330278
2,000229ec398224ef6ca0657da4fc703e,2018-01-22 13:19:16,2018-02-05,-13.444954
3,00024acbcdf0a6daa1e931b038114c75,2018-08-14 13:32:39,2018-08-20,-5.435660
4,00042b26cf59d7ce69dfabb4e55b4fd9,2017-03-01 16:42:31,2017-03-17,-15.303808


In [8]:
print("Rows:", len(delay_df))
print("Columns:", delay_df.columns.tolist())
print(delay_df["delivery_delay"].describe())

Rows: 96476
Columns: ['order_id', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_delay']
count    96476.000000
mean       -11.179120
std         10.186113
min       -146.016123
25%        -16.244384
50%        -11.948941
75%         -6.390000
max        188.975081
Name: delivery_delay, dtype: float64


customer order count

In [9]:
query = """
SELECT
    customer_id,
    COUNT(order_id) AS customer_order_count
FROM orders
GROUP BY customer_id
"""

customer_order_count_df = pd.read_sql(query, conn)

customer_order_count_df.head()

C:\Users\Abcom\AppData\Local\Temp\ipykernel_724\2779002385.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  customer_order_count_df = pd.read_sql(query, conn)


,customer_id,customer_order_count
0,00012a2ce6f8dcda20d059ce98491703,1
1,000161a058600d5901f007fab4c27140,1
2,0001fd6190edaaf884bcaf3d49edf079,1
3,0002414f95344307404f0ace7a26f1d5,1
4,000379cdec625522490c315e70c7a9fb,1


In [10]:
print("Rows:", len(customer_order_count_df))
print("Columns:", customer_order_count_df.columns.tolist())
print(customer_order_count_df["customer_order_count"].describe())

Rows: 99441
Columns: ['customer_id', 'customer_order_count']
count    99441.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
Name: customer_order_count, dtype: float64


In [11]:
query = """
SELECT
    c.customer_unique_id,
    COUNT(o.order_id) AS customer_order_count
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
GROUP BY c.customer_unique_id
"""

customer_order_count_df = pd.read_sql(query, conn)

customer_order_count_df.head()

C:\Users\Abcom\AppData\Local\Temp\ipykernel_724\3489666236.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  customer_order_count_df = pd.read_sql(query, conn)


,customer_unique_id,customer_order_count
0,248ffe10d632bebe4f7267f1f44844c9,1
1,b0015e09bb4b6e47c52844fab5fb6638,1
2,94b11d37cd61cb2994a194d11f89682b,1
3,4893ad4ea28b2c5b3ddf4e82e79db9e6,1
4,0b83f73b19c2019e182fd552c048a22c,1


In [12]:
print("Rows:", len(customer_order_count_df))
print("Columns:", customer_order_count_df.columns.tolist())
print(customer_order_count_df["customer_order_count"].describe())

Rows: 96096
Columns: ['customer_unique_id', 'customer_order_count']
count    96096.000000
mean         1.034809
std          0.214384
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         17.000000
Name: customer_order_count, dtype: float64


customer total spending

In [13]:
query = """
SELECT
    c.customer_unique_id,
    ROUND(SUM(p.payment_value), 2) AS customer_total_spending
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
JOIN order_payments p
    ON o.order_id = p.order_id
GROUP BY c.customer_unique_id
"""

customer_total_spending_df = pd.read_sql(query, conn)

customer_total_spending_df.head()

C:\Users\Abcom\AppData\Local\Temp\ipykernel_724\3143362405.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  customer_total_spending_df = pd.read_sql(query, conn)


,customer_unique_id,customer_total_spending
0,248ffe10d632bebe4f7267f1f44844c9,114.74
1,b0015e09bb4b6e47c52844fab5fb6638,67.41
2,94b11d37cd61cb2994a194d11f89682b,195.42
3,4893ad4ea28b2c5b3ddf4e82e79db9e6,179.35
4,0b83f73b19c2019e182fd552c048a22c,107.01


In [14]:
print("Rows:", len(customer_total_spending_df))
print("Columns:", customer_total_spending_df.columns.tolist())
print(customer_total_spending_df["customer_total_spending"].describe())

Rows: 96093
Columns: ['customer_unique_id', 'customer_total_spending']
count    96093.000000
mean       166.595730
std        231.431028
min          0.000000
25%         63.120000
50%        108.000000
75%        183.530000
max      13664.080000
Name: customer_total_spending, dtype: float64


average order value

In [15]:
query = """
SELECT
    c.customer_unique_id,
    ROUND(AVG(order_total.total_order_value), 2) AS average_order_value
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
JOIN (
    SELECT
        order_id,
        SUM(payment_value) AS total_order_value
    FROM order_payments
    GROUP BY order_id
) order_total
    ON o.order_id = order_total.order_id
GROUP BY c.customer_unique_id
"""

average_order_value_df = pd.read_sql(query, conn)

average_order_value_df.head()

C:\Users\Abcom\AppData\Local\Temp\ipykernel_724\680825382.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  average_order_value_df = pd.read_sql(query, conn)


,customer_unique_id,average_order_value
0,871766c5855e863f6eccc05f988b23cb,72.19
1,eb28e67c4c0b83846050ddfb8a35d051,142.28
2,3818d81c6709e39d06b2738a8d3a2474,216.87
3,af861d436cfc08b2c2ddefd0ba074622,25.78
4,64b576fb70d441e8f1b2d7d446e483c5,218.04


In [16]:
print("Rows:", len(average_order_value_df))
print("Columns:", average_order_value_df.columns.tolist())
print(average_order_value_df["average_order_value"].describe())

Rows: 96093
Columns: ['customer_unique_id', 'average_order_value']
count    96093.000000
mean       161.403253
std        222.310118
min          0.000000
25%         62.460000
50%        105.830000
75%        177.210000
max      13664.080000
Name: average_order_value, dtype: float64


seller revenue

In [17]:
query = """
SELECT
    seller_id,
    ROUND(SUM(price), 2) AS seller_revenue
FROM order_items
GROUP BY seller_id
"""

seller_revenue_df = pd.read_sql(query, conn)

seller_revenue_df.head()

C:\Users\Abcom\AppData\Local\Temp\ipykernel_724\3447113520.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  seller_revenue_df = pd.read_sql(query, conn)


,seller_id,seller_revenue
0,0015a82c2db000af6aaaf3ae2ecb0532,2685.00
1,001cca7ae9ae17fb1caed9dfb1094831,25080.03
2,001e6ad469a905060d959994f1b41e4f,250.00
3,002100f778ceb8431b7a1020ff7ab48f,1234.50
4,003554e2dce176b5555353e4f3555ac8,120.00


In [18]:
print("Rows:", len(seller_revenue_df))
print("Columns:", seller_revenue_df.columns.tolist())
print(seller_revenue_df["seller_revenue"].describe())

Rows: 3095
Columns: ['seller_id', 'seller_revenue']
count      3095.000000
mean       4391.484233
std       13921.997192
min           3.500000
25%         208.850000
50%         821.480000
75%        3280.830000
max      229472.630000
Name: seller_revenue, dtype: float64


seller order count

In [19]:
query = """
SELECT
    seller_id,
    COUNT(DISTINCT order_id) AS seller_order_count
FROM order_items
GROUP BY seller_id
"""

seller_order_count_df = pd.read_sql(query, conn)

seller_order_count_df.head()

C:\Users\Abcom\AppData\Local\Temp\ipykernel_724\2036493079.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  seller_order_count_df = pd.read_sql(query, conn)


,seller_id,seller_order_count
0,0015a82c2db000af6aaaf3ae2ecb0532,3
1,001cca7ae9ae17fb1caed9dfb1094831,200
2,001e6ad469a905060d959994f1b41e4f,1
3,002100f778ceb8431b7a1020ff7ab48f,51
4,003554e2dce176b5555353e4f3555ac8,1


In [20]:
print("Rows:", len(seller_order_count_df))
print("Columns:", seller_order_count_df.columns.tolist())
print(seller_order_count_df["seller_order_count"].describe())

Rows: 3095
Columns: ['seller_id', 'seller_order_count']
count    3095.000000
mean       32.313409
std       105.139763
min         1.000000
25%         2.000000
50%         6.000000
75%        21.500000
max      1854.000000
Name: seller_order_count, dtype: float64


repeat customer indicator

In [21]:
query = """
SELECT
    customer_unique_id,
    CASE
        WHEN COUNT(DISTINCT order_id) > 1 THEN 1
        ELSE 0
    END AS repeat_customer_indicator
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
GROUP BY customer_unique_id
"""

repeat_customer_df = pd.read_sql(query, conn)

repeat_customer_df.head()

C:\Users\Abcom\AppData\Local\Temp\ipykernel_724\4150852899.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  repeat_customer_df = pd.read_sql(query, conn)


,customer_unique_id,repeat_customer_indicator
0,0000366f3b9a7992bf8c76cfdf3221e2,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,0
2,0000f46a3911fa3c0805444483337064,0
3,0000f6ccb0745a6a4b88665a16c9f078,0
4,0004aac84e0df4da2b147fca70cf8255,0


In [22]:
print("Rows:", len(repeat_customer_df))
print("Columns:", repeat_customer_df.columns.tolist())
print(repeat_customer_df["repeat_customer_indicator"].value_counts())

Rows: 96096
Columns: ['customer_unique_id', 'repeat_customer_indicator']
repeat_customer_indicator
0    93099
1     2997
Name: count, dtype: int64
